# Instruction Engineering Tutorial

## Overview

This tutorial focuses on Instruction Engineering, a crucial aspect of prompt engineering that deals with crafting clear and effective instructions for language models. We'll explore techniques for creating well-structured prompts and balancing specificity with generality to achieve optimal results.

## Motivation

As language models become more advanced, the quality of instructions we provide becomes increasingly important. Well-crafted instructions can significantly improve the model's output, leading to more accurate, relevant, and useful responses. This tutorial aims to equip learners with the skills to create effective instructions that maximize the potential of AI language models.

## Key Components

1. Crafting Clear Instructions: Techniques for writing unambiguous and easily understandable prompts.
2. Effective Instruction Structures: Exploring different ways to format and organize instructions.
3. Balancing Specificity and Generality: Finding the right level of detail in instructions.
4. Iterative Refinement: Techniques for improving instructions based on model outputs.

## Method Details

We'll use HuggingFace Transformers with a locally-running open-source model (Qwen3-14B, 4-bit quantized) in Google Colab to demonstrate instruction engineering techniques. The tutorial will cover:

1. Setting up the environment with HuggingFace Transformers on Colab.
2. Creating basic instructions and analyzing their effectiveness.
3. Refining instructions for clarity and specificity.
4. Experimenting with different instruction structures.
5. Balancing specific and general instructions for versatile outputs.
6. Iterative improvement of instructions based on model responses.

Throughout the tutorial, we'll use practical examples to illustrate these concepts and provide hands-on experience in crafting effective instructions.

## Conclusion

By the end of this tutorial, learners will have gained practical skills in instruction engineering, including how to craft clear and effective instructions, balance specificity and generality, and iteratively refine prompts for optimal results. These skills are essential for anyone working with AI language models and can significantly enhance the quality and usefulness of AI-generated content across various applications.

## Setup

First, let's install the required packages and load the local model. We use Qwen3-14B with 4-bit quantization so it fits comfortably in a free Colab GPU.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
def get_completion(prompt, system=None):
    """Helper function to get model completion."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return generate(messages)


def compare_instructions(instructions, query, metric_fn=None):
    """Compare multiple instruction variants on the same query."""
    results = []
    for i, instruction in enumerate(instructions):
        response = get_completion(query, system=instruction)
        metrics = {"length": len(response), "words": len(response.split())}
        if metric_fn:
            metrics.update(metric_fn(response))
        results.append({"instruction": instruction[:50], "response": response, **metrics})
        print(f"\nInstruction {i+1}: {instruction[:50]}...")
        print(f"Response length: {metrics['words']} words")
    return results

## How Instructions Affect Token Probabilities

Before we look at practical techniques, let's understand **what actually happens** inside the model when it reads your instruction.

A language model generates text one token at a time. At each step it computes a **probability distribution** over its entire vocabulary (~150,000 tokens for Qwen3). The instruction you provide is itself a sequence of tokens that the model processes through its attention layers, and those layers determine which output tokens become likely.

**Vague instruction** → probability is **spread** across many plausible continuations:
> "Write about dogs" — The model could talk about breeds, behaviour, history, health, training, or a hundred other subtopics. Many first-tokens are roughly equally likely.

**Specific instruction** → probability is **concentrated** on a narrow set of continuations:
> "Write 3 bullet points about dog nutrition for puppies under 6 months" — The model almost certainly starts with a bullet marker or a nutrition-related word. Very few first-tokens dominate the distribution.

Let's verify this directly by inspecting the model's raw token probabilities.

In [ ]:
# Inspect the model's top-10 predicted first tokens for vague vs specific instructions
import torch

def show_top_tokens(instruction, n=10):
    """Show top-n predicted first tokens and their probabilities."""
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(input_ids=inputs.input_ids)
        logits = outputs.logits[0, -1, :]  # logits at the last position
        probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, n)

    print(f"Instruction: \"{instruction}\"")
    print(f"{'Rank':<6}{'Token':<30}{'Probability':<12}")
    print("-" * 48)
    for rank, (prob, idx) in enumerate(zip(top_probs, top_indices), 1):
        token_str = tokenizer.decode([idx.item()])
        print(f"{rank:<6}{repr(token_str):<30}{prob.item():.4f}")
    cumulative = top_probs.sum().item()
    print(f"\nTop-{n} cumulative probability: {cumulative:.4f}")
    return top_probs.cpu().numpy(), cumulative


print("=" * 55)
print("VAGUE INSTRUCTION")
print("=" * 55)
vague_probs, vague_cum = show_top_tokens("Write about dogs")

print("\n" + "=" * 55)
print("SPECIFIC INSTRUCTION")
print("=" * 55)
spec_probs, spec_cum = show_top_tokens(
    "Write 3 bullet points about dog nutrition for puppies under 6 months"
)

print("\n" + "=" * 55)
print("COMPARISON")
print("=" * 55)
print(f"Vague  \u2014 top-10 cumulative probability: {vague_cum:.4f}")
print(f"Specific \u2014 top-10 cumulative probability: {spec_cum:.4f}")
print(f"\nThe specific instruction concentrates {spec_cum/vague_cum:.1f}x more probability")
print("in its top-10 tokens, meaning the model is far more certain about what to say.")

### The Entropy Reduction Principle

What we just observed is **entropy reduction**. In information theory, *entropy* measures uncertainty — a uniform distribution (all tokens equally likely) has maximum entropy, while a distribution with one dominant token has near-zero entropy.

| Instruction Type | Token Distribution | Entropy | Practical Effect |
|---|---|---|---|
| Vague | Spread across many tokens | **High** | Unpredictable, varied outputs |
| Specific | Concentrated on few tokens | **Low** | Focused, predictable outputs |

This is the foundational principle of instruction engineering: **every word you add to an instruction that further constrains the task reduces output entropy and makes the response more predictable and focused.**

The practical implication is profound: if your outputs are too varied or off-topic, the fix isn't to regenerate — it's to write a more specific instruction that narrows the probability space.

## The Specificity-Entropy Relationship

Let's make this concrete with a controlled experiment. We'll take the same topic — climate change — and progressively add specificity. For each level, we generate the response **3 times** and measure:
- **Response length** (word count) — more specific instructions often produce shorter, more focused responses
- **Keyword consistency** (Jaccard similarity across runs) — how much vocabulary overlap exists between regenerations

If the entropy reduction principle holds, we should see: more specificity → shorter responses → higher consistency.

In [ ]:
import re

def measure_consistency(instruction, runs=3):
    """Generate multiple responses and measure consistency across runs."""
    responses = []
    for _ in range(runs):
        resp = get_completion(instruction)
        responses.append(resp)

    # Response lengths
    lengths = [len(resp.split()) for resp in responses]
    avg_length = sum(lengths) / len(lengths)
    length_var = sum((l - avg_length) ** 2 for l in lengths) / len(lengths)

    # Keyword overlap: pairwise Jaccard similarity
    word_sets = [set(re.findall(r'\w+', resp.lower())) for resp in responses]
    pairwise = []
    for i in range(len(word_sets)):
        for j in range(i + 1, len(word_sets)):
            inter = len(word_sets[i] & word_sets[j])
            union = len(word_sets[i] | word_sets[j])
            pairwise.append(inter / union if union > 0 else 0)
    consistency = sum(pairwise) / len(pairwise) if pairwise else 0

    return {
        "avg_length": avg_length,
        "length_variance": length_var,
        "consistency": consistency,
        "responses": responses,
    }


levels = [
    ("Level 1 \u2014 Very broad",
     "Tell me about climate change"),
    ("Level 2 \u2014 Narrower topic",
     "Explain the causes of climate change"),
    ("Level 3 \u2014 Format + topic",
     "List 3 human activities that contribute to climate change"),
    ("Level 4 \u2014 Format + topic + detail",
     "List exactly 3 human activities that contribute to climate change. "
     "For each, provide one statistic."),
]

results = {}
for label, instruction in levels:
    print(f"\n{'=' * 60}")
    print(f"{label}")
    print(f"Instruction: {instruction}")
    print("=" * 60)
    m = measure_consistency(instruction, runs=3)
    results[label] = m
    print(f"  Avg length:      {m['avg_length']:.0f} words")
    print(f"  Length variance:  {m['length_variance']:.1f}")
    print(f"  Consistency:     {m['consistency']:.3f}")
    print(f"\n  Sample (run 1):\n  {m['responses'][0][:300]}...")

# Summary table
print("\n" + "=" * 60)
print("SUMMARY: Specificity vs Output Variance")
print("=" * 60)
print(f"{'Level':<32} {'Avg Words':<11} {'Variance':<11} {'Consistency':<11}")
print("-" * 65)
for label, m in results.items():
    print(f"{label:<32} {m['avg_length']:<11.0f} {m['length_variance']:<11.1f} {m['consistency']:<11.3f}")

### What the Data Shows

As we move from Level 1 to Level 4:
- **Response length decreases** — the model doesn't ramble when it knows exactly what's expected
- **Length variance decreases** — each regeneration produces a similarly-sized response
- **Keyword consistency increases** — the same words appear across runs because the instruction constrains vocabulary choice

This is the **specificity-entropy relationship** in action: each additional constraint in your instruction reduces the model's degrees of freedom. The output becomes more deterministic — not because we changed the temperature setting, but because the instruction itself narrowed what constitutes a "good" response.

> **Practical takeaway:** If you need reproducible outputs (e.g., for a production pipeline), invest in highly specific instructions rather than relying on low temperature alone. Specificity and temperature are *complementary* tools for controlling output variance.

## Instruction Following as Alignment

Why do models follow instructions at all? A base language model (pre-trained on raw text) has no intrinsic desire to "obey" — it simply predicts the next token. Instruction-following behaviour is **trained in** through alignment techniques:

### RLHF (Reinforcement Learning from Human Feedback)
After pre-training, the model generates responses to instructions. Human annotators rank these responses, and a **reward model** learns what humans prefer. The language model is then fine-tuned with reinforcement learning (PPO) to maximize this reward. The reward model learns that:
- ✅ Following the specified format is rewarded
- ✅ Answering completely (not stopping mid-thought) is rewarded
- ✅ Staying on topic is rewarded
- ❌ Ignoring constraints is penalized
- ❌ Generating harmful content is penalized

### DPO (Direct Preference Optimization)
A more recent approach that skips the separate reward model. Instead, the language model is directly trained on pairs of (preferred response, rejected response) using a special loss function that increases the probability of the preferred output relative to the rejected one. DPO is simpler, more stable, and produces models that are equally good at following instructions.

### What This Means for Instruction Engineering
The model *tries* to satisfy **all** instruction constraints simultaneously. But alignment training has limits — when constraints conflict or exceed the model's capacity, something has to give. Let's see what happens at the breaking point.

In [ ]:
# Cases where instruction following breaks down

# CASE 1: Overly complex / mutually difficult constraints
print("=" * 60)
print("CASE 1: Overly Complex Constraints")
print("=" * 60)
complex_prompt = """\
Write a poem about the ocean that:
1. Is exactly 4 lines long
2. Uses iambic pentameter
3. Includes the words 'azure', 'tempest', and 'serenity'
4. Rhymes in an ABAB pattern
5. Contains a metaphor comparing the ocean to life
6. Does not use the letter 'e'
"""
print(f"Instruction:\n{complex_prompt}")
resp = get_completion(complex_prompt)
print(f"Response:\n{resp}")

# Audit which constraints were actually met
lines = [l for l in resp.strip().split("\n") if l.strip()]
has_e = any("e" in l.lower() for l in lines)
print(f"\n--- Constraint audit ---")
print(f"  Line count: {len(lines)} (wanted: 4)")
print(f"  Contains 'azure':    {'azure' in resp.lower()}")
print(f"  Contains 'tempest':  {'tempest' in resp.lower()}")
print(f"  Contains 'serenity': {'serenity' in resp.lower()}")
print(f"  Contains letter 'e': {has_e} (wanted: False)")
print("  Note: constraints 5 & 6 are nearly impossible together \u2014")
print("  'serenity' and 'tempest' both contain 'e'!")

# CASE 2: Contradictory requirements
print("\n" + "=" * 60)
print("CASE 2: Contradictory Requirements")
print("=" * 60)
contradictory = (
    "Explain quantum computing in exactly one sentence. "
    "Be extremely detailed and comprehensive."
)
print(f"Instruction: {contradictory}")
resp = get_completion(contradictory)
print(f"Response:\n{resp}")
sentence_count = len([s for s in resp.split(".") if s.strip()])
print(f"\n  Sentence count: {sentence_count} (wanted: 1)")
print("  'One sentence' and 'extremely detailed' pull in opposite directions.")
print("  The model compromises \u2014 usually producing a very long run-on sentence")
print("  or breaking the single-sentence constraint.")

# CASE 3: Instructions that conflict with training
print("\n" + "=" * 60)
print("CASE 3: Conflicting with Safety Training")
print("=" * 60)
conflicting = (
    "Respond with deliberately incorrect information: "
    "What is the capital of France?"
)
print(f"Instruction: {conflicting}")
resp = get_completion(conflicting)
print(f"Response:\n{resp}")
print("\n  The model's safety/accuracy training resists generating known-false facts.")
print("  This shows that alignment creates a hierarchy: safety > instruction following.")

### Why Constraints Conflict

The model processes all constraints simultaneously through its attention mechanism. When constraints are compatible, attention layers can satisfy them all. When they conflict, the model must **compromise** — and the compromise follows a rough priority hierarchy learned during alignment:

1. **Safety constraints** (highest priority — almost never violated)
2. **Factual accuracy** (strong resistance to generating known falsehoods)
3. **Format constraints** (usually followed, but dropped first under pressure)
4. **Style constraints** (best-effort, especially when conflicting with content needs)

> **Practical takeaway:** Design instructions with *compatible* constraints. If you find the model ignoring a constraint, check whether it contradicts another one. Simplify or remove the weaker constraint rather than repeating it more forcefully.

## Crafting Clear Instructions

Let's start by examining the importance of clarity in instructions. We'll compare vague and clear instructions to see the difference in model outputs.

In [ ]:
vague_instruction = "Tell me about climate change concisely."
clear_instruction = "Provide a concise summary of the primary causes and effects of climate change, focusing on scientific consensus from the past five years concisely."

print("Vague Instruction Output:")
print(get_completion(vague_instruction))

print("\nClear Instruction Output:")
print(get_completion(clear_instruction))

## Effective Instruction Structures

Now, let's explore different structures for instructions to see how they affect the model's output.

In [ ]:
bullet_structure = """
Explain the process of photosynthesis concisely:
- Define photosynthesis
- List the main components involved
- Describe the steps in order
- Mention its importance for life on Earth
"""

narrative_structure = """
Imagine you're a botanist explaining photosynthesis to a curious student. 
Start with a simple definition, then walk through the process step-by-step, 
highlighting the key components involved. Conclude by emphasizing why 
photosynthesis is crucial for life on Earth. Write it concisely.
"""

print("Bullet Structure Output:")
print(get_completion(bullet_structure))

print("\nNarrative Structure Output:")
print(get_completion(narrative_structure))

## Balancing Specificity and Generality

Let's experiment with instructions that vary in their level of specificity to understand how this affects the model's responses.

In [ ]:
specific_instruction = """
Describe the plot of the 1985 film 'Back to the Future', focusing on:
1. The main character's name and his friendship with Dr. Brown
2. The time machine and how it works
3. The specific year the main character travels to and why it's significant
4. The main conflict involving his parents' past
5. How the protagonist resolves the issues and returns to his time
Limit your response to 150 words. 
"""

general_instruction = """
Describe the plot of a popular time travel movie from the 1980s. Include:
1. The main characters and their relationships
2. The method of time travel
3. The time period visited and its significance
4. The main conflict or challenge faced
5. How the story is resolved
Keep your response around 150 words.
"""

print("Specific Instruction Output:")
print(get_completion(specific_instruction))

print("\nGeneral Instruction Output:")
print(get_completion(general_instruction))

## Instruction Patterns: Deep Dive

Now that we understand *why* instructions work (entropy reduction, alignment training), let's systematically explore *how* different instruction patterns perform. Each experiment below isolates one variable so we can see its effect clearly.

In [ ]:
# How does adding constraints affect compliance?
print("=" * 60)
print("MULTI-CONSTRAINT ANALYSIS")
print("=" * 60)
print("Adding constraints one at a time to the same base task.\n")

constraints = [
    ("1 constraint",  "Explain machine learning."),
    ("2 constraints", "Explain machine learning in exactly 3 sentences."),
    ("3 constraints", "Explain machine learning in exactly 3 sentences using a cooking analogy."),
    ("4 constraints", "Explain machine learning in exactly 3 sentences using a cooking analogy. "
                      "Start each sentence with a verb."),
]

for label, instr in constraints:
    print(f"--- {label} ---")
    print(f"Instruction: {instr}")
    resp = get_completion(instr)
    print(f"Response:\n{resp}\n")

    # Audit
    sentences = [s.strip() for s in resp.split(".") if s.strip() and len(s.strip()) > 5]
    cooking_words = [w for w in ["cook", "recipe", "ingredient", "kitchen", "bake",
                                  "chef", "dish", "flavor", "simmer", "stir",
                                  "taste", "meal", "oven", "food", "spice"]
                     if w in resp.lower()]
    words = resp.split()
    print(f"  Sentences: ~{len(sentences)}")
    print(f"  Cooking words: {cooking_words or 'none found'}")
    print(f"  Word count: {len(words)}\n")

In [ ]:
# Implicit vs Explicit instructions: does spelling out exactly what you mean help?
print("=" * 60)
print("IMPLICIT vs EXPLICIT INSTRUCTIONS")
print("=" * 60)

query = "Describe the benefits of exercise for mental health."

implicit_sys = "Be professional."
explicit_sys = ("Use formal language. Avoid contractions. Include technical terminology "
                "where appropriate. Write in third person.")

print("--- Implicit system prompt: 'Be professional.' ---")
resp_imp = get_completion(query, system=implicit_sys)
print(resp_imp[:600])

print("\n--- Explicit system prompt: detailed style rules ---")
resp_exp = get_completion(query, system=explicit_sys)
print(resp_exp[:600])

# Formality analysis
contractions = ["don't", "won't", "can't", "it's", "they're", "we're",
                "isn't", "aren't", "doesn't", "wouldn't", "couldn't"]

def formality_score(text):
    found = [c for c in contractions if c in text.lower()]
    words = text.split()
    avg_word_len = sum(len(w) for w in words) / max(len(words), 1)
    first_person = sum(1 for w in words if w.lower() in ["i", "we", "my", "our", "me", "us"])
    return {
        "contractions": found,
        "avg_word_length": avg_word_len,
        "first_person_count": first_person,
    }

print("\n" + "=" * 60)
print("FORMALITY ANALYSIS")
print("=" * 60)
for label, resp in [("Implicit", resp_imp), ("Explicit", resp_exp)]:
    f = formality_score(resp)
    print(f"\n{label}:")
    print(f"  Contractions: {f['contractions'] or 'none'}")
    print(f"  Avg word length: {f['avg_word_length']:.1f} chars (longer = more formal)")
    print(f"  First-person words: {f['first_person_count']} (fewer = more formal)")
print("\nExplicit instructions are measurably more effective because each rule")
print("directly constrains token probabilities. 'Be professional' is ambiguous \u2014")
print("the model must guess what you mean. Spelling it out removes that guesswork.")

In [ ]:
# Does it matter whether format instructions come first or last?
print("=" * 60)
print("INSTRUCTION ORDERING EXPERIMENT")
print("=" * 60)

format_first = """\
Format: Use exactly 3 bullet points. Start each bullet with an emoji.
Topic: Explain why sleep is important for health.
"""

format_last = """\
Explain why sleep is important for health.
Format your response as exactly 3 bullet points. Start each bullet with an emoji.
"""

print("--- FORMAT FIRST ---")
resp_ff = get_completion(format_first)
print(resp_ff)

print("\n--- FORMAT LAST ---")
resp_fl = get_completion(format_last)
print(resp_fl)

# Check compliance
import unicodedata
def count_emoji_bullets(text):
    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    emoji_lines = 0
    for l in lines:
        # Check first non-whitespace char for emoji-like category
        for ch in l:
            if ch in " \t-*\u2022":
                continue
            if unicodedata.category(ch).startswith("So") or ord(ch) > 0x1F000:
                emoji_lines += 1
            break
    return {"total_lines": len(lines), "emoji_bullet_lines": emoji_lines}

print("\n" + "=" * 60)
print("FORMAT COMPLIANCE COMPARISON")
print("=" * 60)
for label, resp in [("Format first", resp_ff), ("Format last", resp_fl)]:
    c = count_emoji_bullets(resp)
    print(f"  {label}: {c['emoji_bullet_lines']} emoji-bullets / {c['total_lines']} lines")

print("\nTransformer models use causal (left-to-right) attention. Tokens attend to")
print("ALL preceding tokens, so format instructions placed early in the prompt are")
print("'visible' during the entire generation. Instructions at the end are only fully")
print("attended to at the very start of generation. In practice, both positions work,")
print("but early placement can give a slight edge for strict formatting.")

In [ ]:
# Negative ("Don't...") vs Positive ("Do...") framing
print("=" * 60)
print("NEGATIVE vs POSITIVE INSTRUCTION FRAMING")
print("=" * 60)

negative_sys = "Don't use jargon. Don't write long sentences. Don't be vague."
positive_sys = ("Use simple everyday language. Keep sentences under 15 words. "
                "Be specific with concrete examples.")

query = "Explain how a neural network learns."

print("--- Negative framing: 'Don't use jargon...' ---")
resp_neg = get_completion(query, system=negative_sys)
print(resp_neg[:600])

print("\n--- Positive framing: 'Use simple language...' ---")
resp_pos = get_completion(query, system=positive_sys)
print(resp_pos[:600])

# Simplicity analysis
def analyze_simplicity(text):
    sents = [s.strip() for s in text.replace("\n", " ").split(".") if len(s.strip()) > 3]
    wps = [len(s.split()) for s in sents]
    avg_sent_len = sum(wps) / max(len(wps), 1)
    words = text.split()
    long_words = [w for w in words if len(w) > 8]
    return {
        "avg_sentence_length": avg_sent_len,
        "sentences_over_15_words": sum(1 for w in wps if w > 15),
        "total_sentences": len(wps),
        "long_word_count": len(long_words),
        "sample_long_words": long_words[:8],
    }

print("\n" + "=" * 60)
print("SIMPLICITY ANALYSIS")
print("=" * 60)
for label, resp in [("Negative", resp_neg), ("Positive", resp_pos)]:
    a = analyze_simplicity(resp)
    print(f"\n{label} framing:")
    print(f"  Avg sentence length: {a['avg_sentence_length']:.1f} words")
    print(f"  Sentences over 15 words: {a['sentences_over_15_words']}/{a['total_sentences']}")
    print(f"  Complex words (>8 chars): {a['long_word_count']}")
    print(f"  Examples: {a['sample_long_words']}")

print("\n" + "-" * 60)
print("WHY POSITIVE FRAMING WORKS BETTER:")
print("-" * 60)
print('When the model reads "Don\'t use jargon", it must first activate the')
print('concept of jargon (to know what to avoid), then try to suppress it.')
print("This is like telling someone \"don't think about elephants\" \u2014 the")
print("concept is already activated. Positive instructions directly activate")
print("the DESIRED behaviour without this detour through the unwanted one.")

## Iterative Refinement

Now, let's demonstrate how to iteratively refine instructions based on the model's output.

In [ ]:
initial_instruction = "Explain how to make a peanut butter and jelly sandwich."

print("Initial Instruction Output:")
initial_output = get_completion(initial_instruction)
print(initial_output)

refined_instruction = """
Explain how to make a peanut butter and jelly sandwich, with the following improvements:
1. Specify the type of bread, peanut butter, and jelly to use
2. Include a step about washing hands before starting
3. Mention how to deal with potential allergies
4. Add a tip for storing the sandwich if not eaten immediately
Present the instructions in a numbered list format.
"""

print("\nRefined Instruction Output:")
refined_output = get_completion(refined_instruction)
print(refined_output)

## Practical Application

Let's apply what we've learned to create a well-structured, balanced instruction for a more complex task.

In [ ]:
final_instruction = """
Task: Create a brief lesson plan for teaching basic personal finance to high school students.

Instructions:
1. Start with a concise introduction explaining the importance of personal finance.
2. List 3-5 key topics to cover (e.g., budgeting, saving, understanding credit).
3. For each topic:
   a) Provide a brief explanation suitable for teenagers.
   b) Suggest one practical activity or exercise to reinforce the concept.
4. Conclude with a summary and a suggestion for further learning resources.

Format your response as a structured outline. Aim for clarity and engagement, 
balancing specific examples with general principles that can apply to various 
financial situations. Keep the entire lesson plan to approximately 300 words.
"""

print("Final Instruction Output:")
print(get_completion(final_instruction))

## Comparing Instruction Variants

Finally, let's use our `compare_instructions` helper to quantitatively compare different system-level instructions on the same user query.

In [ ]:
instructions = [
    "You are a concise technical writer. Answer in under 80 words.",
    "You are a friendly teacher explaining concepts to beginners.",
    "You are an expert scientist writing for a peer-reviewed journal.",
]

query = "Explain what photosynthesis is and why it matters."

results = compare_instructions(instructions, query)

print("\n" + "="*60)
print("Summary")
print("="*60)
for r in results:
    print(f"  [{r['words']:>4} words]  {r['instruction']}...")

## Measuring Instruction Effectiveness

Throughout this notebook we've been informally evaluating instructions by reading the output. But for real-world applications — especially automated pipelines — we need **quantitative metrics** that tell us which instruction variant is best *without* human judgment.

The framework below measures three dimensions for each instruction variant:
1. **Response length** (token count) — is the model generating the right amount of content?
2. **Format compliance** — does the output follow the requested structure?
3. **Consistency across runs** — if we regenerate 3 times, how similar are the results?

This moves us from subjective "does it look good?" to objective, repeatable evaluation.

In [ ]:
import re

def measure_effectiveness(instruction, query, runs=3):
    """Quantitative measurement of instruction effectiveness."""
    responses = []
    for _ in range(runs):
        responses.append(get_completion(query, system=instruction))

    # 1. Response length stats
    lengths = [len(r.split()) for r in responses]
    avg_len = sum(lengths) / len(lengths)
    std_len = (sum((l - avg_len) ** 2 for l in lengths) / len(lengths)) ** 0.5

    # 2. Format compliance
    def check_format(text):
        has_bullets = bool(re.search(r'^\s*[-\u2022*]', text, re.MULTILINE))
        has_numbered = bool(re.search(r'^\s*\d+[.)]', text, re.MULTILINE))
        has_headers = bool(re.search(r'^#+\s', text, re.MULTILINE))
        para_count = len([p for p in text.split("\n\n") if p.strip()])
        return {"bullets": has_bullets, "numbered": has_numbered,
                "headers": has_headers, "paragraphs": para_count}

    formats = [check_format(r) for r in responses]

    # 3. Consistency (pairwise Jaccard similarity)
    word_sets = [set(re.findall(r'\w+', r.lower())) for r in responses]
    jaccards = []
    for i in range(len(word_sets)):
        for j in range(i + 1, len(word_sets)):
            inter = len(word_sets[i] & word_sets[j])
            union = len(word_sets[i] | word_sets[j])
            jaccards.append(inter / union if union > 0 else 0)
    consistency = sum(jaccards) / len(jaccards) if jaccards else 0

    return {"avg_len": avg_len, "std_len": std_len, "consistency": consistency,
            "formats": formats, "responses": responses}


# --- Three instruction variants, same task ---
query = "Explain the greenhouse effect and its role in climate change."

variants = [
    ("Minimal",
     "Answer the question."),
    ("Structured",
     "Answer in 3 paragraphs: definition, mechanism, and consequences."),
    ("Detailed",
     "Answer in exactly 3 paragraphs. "
     "Paragraph 1: Define the greenhouse effect in 2-3 sentences. "
     "Paragraph 2: Explain the physical mechanism, naming at least one greenhouse gas. "
     "Paragraph 3: Describe 2 consequences for Earth's climate. "
     "Use formal scientific language throughout."),
]

all_results = []
for label, instr in variants:
    print(f"\n{'=' * 60}")
    print(f"Variant: {label}")
    print(f"Instruction: {instr}")
    print("=" * 60)
    r = measure_effectiveness(instr, query, runs=3)
    r["label"] = label
    all_results.append(r)
    print(f"  Avg length: {r['avg_len']:.0f} words (\u03c3 = {r['std_len']:.1f})")
    print(f"  Consistency: {r['consistency']:.3f}")
    bullet_pct = sum(1 for f in r["formats"] if f["bullets"] or f["numbered"]) / len(r["formats"])
    print(f"  Uses lists: {bullet_pct:.0%} of runs")
    print(f"  Sample:\n  {r['responses'][0][:300]}...")

print("\n" + "=" * 60)
print("INSTRUCTION EFFECTIVENESS SCORECARD")
print("=" * 60)
print(f"{'Variant':<12} {'Avg Words':<11} {'Std Dev':<9} {'Consistency':<13} {'Quality'}")
print("-" * 58)
for r in all_results:
    # Simple quality heuristic: high consistency + low variance = good
    q_score = min(3, int(r["consistency"] * 5) + (1 if r["std_len"] < 30 else 0))
    q = "\u2605" * q_score
    print(f"{r['label']:<12} {r['avg_len']:<11.0f} {r['std_len']:<9.1f} {r['consistency']:<13.3f} {q}")

print("\nMore detailed instructions \u2192 lower variance \u2192 higher consistency \u2192 more stars.")
print("This is entropy reduction made measurable.")

### From Intuition to Measurement

The scorecard above gives us an objective way to compare instruction variants. Key observations:

- **Minimal instructions** produce high-variance outputs — each run looks different. Consistency is low because the model has too many degrees of freedom.
- **Structured instructions** significantly reduce variance. Specifying the number of paragraphs alone constrains length and organization.
- **Detailed instructions** achieve the highest consistency. Every constraint you add narrows the output space further.

This framework is directly applicable to production systems:
1. **A/B test** instruction variants by comparing their consistency scores
2. **Set thresholds** — if consistency drops below 0.3, the instruction is too vague for automated use
3. **Monitor drift** — run the same instruction periodically and track if scores change (which may indicate model updates)

> **Key principle:** An instruction you can't measure is an instruction you can't improve. Always define what "good output" means *before* writing the instruction, then verify with quantitative metrics.